In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw

import numpy as np
import scipy.optimize


# =========================================================
# GEL MODEL
# =========================================================

class gel_debonded2D:

    def __init__(
        self,
        length=90.0,
        thickness=1.62,
        phi0=0.2,
        mu_bar=-0.03
    ):

        self.L = length
        self.d = thickness

        self.phi0 = phi0

        # thermodynamics

        T = 25 + 273.15
        K_B = 1.380649e-23
        V_m = 3e-29

        self.entropic_unit = K_B*T/V_m*1e-6

        self.gamma = 0.001

        self.chi = 0.4

        self.G = self.gamma*self.entropic_unit

        vapor_pressure = 3.2e-3

        self.p0_bar = vapor_pressure/self.entropic_unit

        self.mu_bar = mu_bar

        self.p_bar = self.p0_bar*np.exp(mu_bar)

        self.mass_density = 1.23e-6

        # ==========================================
        # reference energy
        # ==========================================

        def dH_numpy(J):

            phi = self.phi0/J

            return (
                phi
                + np.log(1-phi)
                + self.chi*phi**2
                - self.gamma/J
                + self.p_bar
                - self.mu_bar
            )

        def aux_iso(s):

            return s*dH_numpy(s**3) + self.gamma

        self.lambda_iso = scipy.optimize.fsolve(
            aux_iso,
            1.2
        )[0]

        def aux_energy(l1, l2, l3):

            J = l1*l2*l3

            phi = self.phi0/J

            return (
                0.5*self.G*(l1**2+l2**2+l3**2-3)
                + self.entropic_unit*(
                    (J-self.phi0)*np.log(1-phi)
                    + self.phi0*self.chi*(1-phi)
                    - self.gamma*np.log(J)
                    + (self.p_bar-self.mu_bar)*(J-self.phi0)
                )
            )

        self.reference_energy_density = aux_energy(
            self.lambda_iso,
            self.lambda_iso,
            self.lambda_iso
        )

    # =====================================================
    # FUNCTIONS
    # =====================================================

    def phi(self, J):

        return self.phi0/J

    def H(self, J):

        phi = self.phi(J)

        eps = 1e-12

        one_minus_phi_safe = IfPos(
            1 - phi - eps,
            1 - phi,
            eps
        )

        J_safe = IfPos(
            J - eps,
            J,
            eps
        )

        return (
            (J-self.phi0)*log(one_minus_phi_safe)
            + self.phi0*self.chi*(1-phi)
            - self.gamma*log(J_safe)
            + (self.p_bar-self.mu_bar)*(J-self.phi0)
        )

    # =====================================================
    # ENERGY
    # =====================================================

    def W(self, F):

        J = Det(F)

        C = F.trans*F

        return (
            0.5*self.G*(Trace(C)-2)
            + self.entropic_unit*self.H(J)
            - self.reference_energy_density
        )


# =========================================================
# MAIN
# =========================================================

L = 90.0

d = 1.62

phi0 = 0.2

order = 3

mu_input = 0.03

mu_bar = -mu_input

first_index_delta = 40
last_index_delta = 50

indexes_deltas = range(
    first_index_delta,
    last_index_delta+1
)

print(indexes_deltas)

# =========================================================
# CREATE GEL
# =========================================================

gel = gel_debonded2D(
    length=L,
    thickness=d,
    phi0=phi0,
    mu_bar=mu_bar
)

gravity = CoefficientFunction((0,-9.8))

# =========================================================
# LOAD DELTAS
# =========================================================

folder_name_suffix = (
    str(int(d))
    + '_'
    + str(int(d%1*100)).zfill(2)
)

delta_values = np.loadtxt(
    'meshes' + folder_name_suffix + '/deltas'
)

# =========================================================
# LOOP OVER MESHES
# =========================================================

for index_delta in indexes_deltas:

    print("")
    print("================================")
    print("Mesh =", index_delta)
    print("================================")

    mesh_file = (
        'meshes'
        + folder_name_suffix
        + '/mesh{}.vol'.format(index_delta)
    )

    mesh = Mesh(mesh_file)

    delta = delta_values[index_delta]

    print("delta =", delta)

    # IMPORTANT:
    # same order as simulation

    fes = VectorH1(
        mesh,
        order=order
    )

    print('nDoF =', fes.ndof)

    # =====================================================
    # FILE NAME
    # =====================================================

    filename_suffix = (
        f'_d={d:.2f}'
        f'_delta={delta:.3f}'
        f'_muBarAbs={abs(mu_bar):.3f}'
    )

    filename = (
        'gridfunctions'
        + folder_name_suffix
        + '/result_debonded2D'
        + filename_suffix
        + f'_order={order}'
    )

    print("Loading:")
    print(filename + ".gfu")

    # =====================================================
    # LOAD SOLUTION
    # =====================================================

    u = GridFunction(fes)

    u.Load(filename + '.gfu')

    # =====================================================
    # ENERGY
    # =====================================================

    F = Id(2) + Grad(u)

    energy_density = gel.W(F)

    avg_energy = Integrate(
        energy_density,
        mesh,
        order=5
    )/(L*d)

    print("Average energy =", avg_energy)

    total_energy = Integrate(
        energy_density
        - gel.mass_density*InnerProduct(gravity,u),
        mesh,
        order=8
    )

    print("Total energy =", total_energy)

    # =====================================================
    # GEOMETRY
    # =====================================================

    deformed_length = (
        u(mesh(L/2,d))[0]
        - u(mesh(-L/2,d))[0]
        + L
    )

    print("Deformed length =", deformed_length)

    aux = np.array(
        u(mesh(L/2,d))
    )

    deformed_thickness = aux[1] + d

    print(
        "Deformed thickness =",
        deformed_thickness
    )

    # =====================================================
    # VISUALIZATION
    # =====================================================

    Draw(
        energy_density,
        mesh,
        deformation=u
    )

    print("")